# AI-Native Compliance Infrastructure — Starter Build
### Phase 0–2: Environment Setup, Data Ingestion, Core RAG Engine

**How to use this notebook:**
1. Upload this file to Kaggle → **Create → New Notebook → File → Import Notebook**
2. On the right sidebar: **Settings → Accelerator → GPU T4 x2** (or P100)
3. Run each cell top to bottom (Shift+Enter)
4. Only uses **public** documents and synthetic data — nothing proprietary, nothing from your employer

**What this notebook builds:**
- A working retrieval engine (RAG) that can answer questions about Medicaid policy documents
- The foundation that Phase 3 (change-detection) and Phase 4 (draft generation) will build on top of

---

## Phase 0 — Environment Setup

We're using small, efficient open models so this runs comfortably on Kaggle's free T4 GPU:
- **Embedding model**: `all-MiniLM-L6-v2` (tiny, fast, great for retrieval)
- **Generation model**: `Qwen2.5-7B-Instruct` in 4-bit quantization (~5GB VRAM, fits easily on a 16GB T4)

Why Qwen2.5-7B: strong instruction-following for its size, handles document Q&A well, and is small enough to stay fast on free-tier hardware. We can upgrade to a bigger model later once you move to a paid GPU (RunPod, etc.)

In [ ]:
# Install dependencies (takes ~2-3 min on first run)
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu pypdf
print("Dependencies installed.")

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU T4 x2, then re-run.")

## Phase 1 — Data Collection (Public Documents Only)

For this starter build, drop a few **public** Medicaid/CMS policy PDFs into Kaggle's `/kaggle/input/` via
**Add Data** in the right sidebar, or paste plain text directly into the `SAMPLE_DOCS` list below to get started
immediately without needing real files yet.

Good public sources to pull from later:
- Medicaid.gov state plan amendments
- CMS provider bulletins (publicly posted)
- Federal Register Medicaid rule updates

For now, this cell includes placeholder sample text so you can run the full pipeline end-to-end today.
Replace with real public policy text as soon as you have it.

In [ ]:
# Placeholder sample documents — REPLACE with real public CMS/Medicaid text
# (Each string simulates one policy document/section)

SAMPLE_DOCS = [
    {
        "id": "policy_prior_auth_v1",
        "title": "Prior Authorization Policy — Outpatient Services (Sample, v1)",
        "text": """Effective January 2026, prior authorization is required for all outpatient
        physical therapy services exceeding 12 visits per calendar year. Providers must submit
        requests at least 5 business days before the 13th visit. Requests submitted without
        supporting clinical documentation will be denied. This policy applies to all managed
        care organizations contracted under the state Medicaid program."""
    },
    {
        "id": "policy_prior_auth_v2",
        "title": "Prior Authorization Policy — Outpatient Services (Sample, v2 - UPDATED)",
        "text": """Effective July 2026, prior authorization is required for all outpatient
        physical therapy services exceeding 20 visits per calendar year, increased from the
        previous threshold of 12 visits. Providers must submit requests at least 3 business
        days before the 21st visit, reduced from the prior 5-day requirement. Requests
        submitted without supporting clinical documentation will be denied. This policy
        applies to all managed care organizations contracted under the state Medicaid program."""
    },
    {
        "id": "policy_telehealth",
        "title": "Telehealth Reimbursement Policy (Sample)",
        "text": """Telehealth visits are reimbursed at parity with in-person visits for
        behavioral health services only. Audio-only visits are reimbursed at 80% of the
        standard rate. Providers must document the modality of service delivery on every claim."""
    },
]

print(f"Loaded {len(SAMPLE_DOCS)} sample documents.")
print("Replace SAMPLE_DOCS with real public policy text when ready.")

In [ ]:
# Optional: load real PDFs if you've added them via Kaggle "Add Data"
import os
from pypdf import PdfReader

def load_pdfs_from_folder(folder_path):
    """Reads every PDF in a folder and returns them in the same format as SAMPLE_DOCS."""
    docs = []
    if not os.path.exists(folder_path):
        print(f"No folder found at {folder_path} — skipping. Using SAMPLE_DOCS instead.")
        return docs
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(".pdf"):
            reader = PdfReader(os.path.join(folder_path, fname))
            text = "\n".join(page.extract_text() or "" for page in reader.pages)
            docs.append({"id": fname, "title": fname, "text": text})
    return docs

# Example: uncomment once you've added a Kaggle dataset with real public PDFs
# real_docs = load_pdfs_from_folder("/kaggle/input/your-dataset-name")
# if real_docs:
#     SAMPLE_DOCS = real_docs
#     print(f"Loaded {len(SAMPLE_DOCS)} real documents instead.")

## Phase 2 — Core RAG Engine

### Step 1: Chunk documents
Long documents need to be split into smaller passages so retrieval is precise.

In [ ]:
def chunk_text(text, chunk_size=300, overlap=50):
    """Simple word-based chunking with overlap so context isn't lost at boundaries."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

# Build the chunk store
all_chunks = []
for doc in SAMPLE_DOCS:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "doc_id": doc["id"],
            "title": doc["title"],
            "chunk_id": f'{doc["id"]}_{i}',
            "text": chunk,
        })

print(f"Created {len(all_chunks)} chunks from {len(SAMPLE_DOCS)} documents.")

### Step 2: Embed chunks and build a searchable index
This turns each text chunk into a vector, so we can find the most relevant chunks for any question.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Index built with {index.ntotal} vectors of dimension {dimension}.")

### Step 3: Retrieval function
Given a question, find the most relevant chunks.

In [ ]:
def retrieve(query, top_k=3):
    query_vec = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, top_k)
    results = [all_chunks[i] for i in indices[0]]
    return results

# Quick test — no LLM needed yet, just checking retrieval quality
test_results = retrieve("What is the prior authorization threshold for physical therapy?")
for r in test_results:
    print(f"[{r['title']}]")
    print(r["text"][:200], "...\n")

### Step 4: Load the generation model (Qwen2.5-7B-Instruct, 4-bit)
This is the model that will read the retrieved chunks and write a plain-language answer.
First load will download ~5GB — takes a few minutes on Kaggle.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded.")

### Step 5: Generation function — combine retrieval + LLM
This is the core of the RAG engine: retrieve relevant policy text, then ask the model to answer
using ONLY that retrieved context (reduces hallucination, keeps answers grounded in real source text).

In [ ]:
def ask(query, top_k=3, max_new_tokens=300):
    retrieved = retrieve(query, top_k=top_k)
    context = "\n\n".join(f"[Source: {r['title']}]\n{r['text']}" for r in retrieved)

    prompt = f"""You are a Medicaid compliance assistant. Answer the question using ONLY the
context provided below. If the context does not contain the answer, say so clearly.
Always cite which source document your answer comes from.

Context:
{context}

Question: {query}

Answer:"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response, retrieved

# Test it
answer, sources = ask("What is the current prior authorization threshold for outpatient physical therapy?")
print("ANSWER:\n", answer)
print("\nSOURCES USED:", [s["title"] for s in sources])

---
## What You've Built So Far

You now have a working RAG engine that:
- Ingests policy documents (sample or real public PDFs)
- Retrieves the most relevant passages for any question
- Generates grounded, cited answers using an open-source 7B model
- Runs entirely on Kaggle's free GPU — no local hardware needed

## Try It Yourself
Run the cell below with your own questions to see the engine in action.

In [ ]:
# Try your own question
my_question = "What changed about telehealth reimbursement?"
answer, sources = ask(my_question)
print("ANSWER:\n", answer)
print("\nSOURCES:", [s["title"] for s in sources])

---
## Next Steps (Phase 3 — coming next)

Notice the two sample documents `policy_prior_auth_v1` and `policy_prior_auth_v2` above — they're
the same policy at two points in time. **Phase 3 builds the "diff" layer**: automatically detecting
that v2 changed the visit threshold from 12→20 and the notice period from 5→3 days, then mapping
that change to which internal SOPs it would affect.

That's the feature that makes this a *compliance* product instead of just a chatbot — save this
notebook (File → Save Version on Kaggle) and we'll build that layer next.